In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import spearmanr

# LOAD DATA

DATA_DIR = Path("./patienten-visiten")
csv_files = sorted(DATA_DIR.glob("Patients_V*.csv"))

dfs = [pd.read_csv(f, sep=";") for f in csv_files]
data = pd.concat(dfs, ignore_index=True)

print("Rows:", len(data))

# CLEAN FUNCTION 

def clean_name(name):
    """
    Removes:
    - _K1, _K2, ...
    - _K
    """

    name = str(name)

    name = name.replace(",", ".")  

    # remove _K12, _K1 etc
    name = pd.Series([name]).str.replace(r"_K\d+$", "", regex=True).iloc[0]

    # remove trailing _K
    name = name.replace("_K", "")

    return name

# SCORE + FACTOR BASE CLEANING

score_cols = [c for c in data.columns if any(
    c.startswith(x) for x in [
        "midas_result", "dass_depression", "dass_fear",
        "dass_stress", "gvas_result", "pgic_result", "chiq_result"
    ]
)]

factor_cols = [c for c in data.columns if any(
    c.startswith(x) for x in [
        "gender", "birthyear", "weight", "pregnant",
        "highest_degree", "family_status", "current_employed",
        "working_hours_model", "shiftwork", "working_hours_reduced",
        "children", "headache_days", "days_medication",
        "intensity", "days_lost", "days_doctor",
        "days_ER", "days_hospital"
    ]
)]

# TYPE DETECTION

def to_numeric(s):
    return pd.to_numeric(
        s.astype(str).str.replace(",", ".", regex=False),
        errors="coerce"
    )

def detect_type(s):

    s = s.dropna()

    if len(s) == 0:
        return "unknown"

    num = to_numeric(s)

    if num.notna().mean() > 0.9:
        if num.nunique() <= 2:
            return "binary"
        return "numeric"

    return "categorical"


# RESULTS

results = []


for score_col in score_cols:

    score_name = clean_name(score_col)
    score = to_numeric(data[score_col])

    for factor_col in factor_cols:

        factor_name = clean_name(factor_col)

        tmp = pd.DataFrame({
            "score": score,
            "factor": data[factor_col]
        }).dropna()

        if len(tmp) < 30:
            continue

        ftype = detect_type(tmp["factor"])

        # NUMERIC
        if ftype == "numeric":

            x = to_numeric(tmp["factor"])
            mask = x.notna()

            if mask.sum() > 30:

                corr, _ = spearmanr(
                    x[mask],
                    tmp.loc[mask, "score"]
                )

                results.append({
                    "score": score_name,
                    "factor": factor_name,
                    "type": "numeric",
                    "value": None,
                    "association": corr,
                    "n": mask.sum()
                })

        # BINARY
        elif ftype == "binary":

            x = to_numeric(tmp["factor"])

            if x.nunique() == 2:

                corr = x.corr(tmp["score"])

                results.append({
                    "score": score_name,
                    "factor": factor_name,
                    "type": "binary",
                    "value": None,
                    "association": corr,
                    "n": len(tmp)
                })

        # CATEGORICAL 
        else:

            global_mean = tmp["score"].mean()

            for cat, group in tmp.groupby("factor"):

                if len(group) < 10:
                    continue

                diff = group["score"].mean() - global_mean

                results.append({
                    "score": score_name,
                    "factor": factor_name,
                    "type": "categorical",
                    "value": cat,
                    "association": diff,
                    "n": len(group)
                })

# OUTPUT

df = pd.DataFrame(results)

df.to_csv(
    "FULL_clean_K_merged_analysis.csv",
    index=False
)

print("Saved: questionnaire_data_other_factors.csv")

print(df.head())

C:\Users\veron\AppData\Local\Temp\ipykernel_38252\3419089454.py:13: DtypeWarning: Columns (106,107,228,230,243,245,247,249,251,252,260,262,264,266,268,269,277,279,281,283,285,286,294,300,302,303,311,313,371,373,374,383,385,386,395,396,397,398,399,407,408,409,410,411,416,418,419,420,421,422,423,428,430,431,432,433,435,440,441,442,443,444,445,446,447,448,450,451,452,453,454,459,460,462,463,464,465,466,470,472,474,475,476,477,484,486,487,488,489,490,496,498,499,501,502,508,510,511,513,730,735,737,755,756,761,762,763,777,779,781,782,788,791,799,801,803,805,808,814,817,825,827,829,1016,1024,1028,1033,1034,1040,1041,1042,1046,1048,1051,1052,1058,1059,1060,1064,1066,1069,1070,1076,1077,1078,1082,1084,1087,1094,1095,1096,1099,1100,1102,1105,1112,1113,1114,1115,1117,1118,1120,1123,1124,1130,1131,1132,1133,1134,1135,1136,1138,1141,1144,1145,1148,1149,1151,1152,1153,1156,1161,1162,1163,1166,1167,1169,1170,1171,1174,1177,1178,1179,1180,1181,1184,1185,1187,1188,1189,1195,1197,1198,1199,1202,1203,12

Rows: 24497
Saved: FULL_clean_K_merged_analysis.csv
             score     factor         type     value  association    n
0  dass_depression     gender  categorical  männlich     0.222559   40
1  dass_depression     gender  categorical  weiblich    -0.026261  339
2  dass_depression  birthyear      numeric      None    -0.060227  379
3  dass_depression     weight      numeric      None     0.116570  379
4  dass_depression   pregnant  categorical      nein     0.027391  336
